# Unitree G1：公式設定の学習を1,000反復から10,000反復へ

これまでにIsaac Gymで **1,000反復まで正常に学習・保存**できました．このノートブックは，Google Driveにある最新のチェックポイントから継続学習します．**初期状態からの再学習は行いません**．

- **標準の再開元**：`Oct23_16-15-05_colab_resume_500_to_1000/model_1000.pt`
- **目標**：累積10,000反復（1,000からなら追加9,000反復）
- **GPU学習**：Colab ProのGPUランタイム（前回はA100／4,096並列環境）
- **保存先**：`MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training`
- **再開**：実行中にColabが切断されたら，このノートブックを開き直して最初からセルを実行すれば，前回までにGoogle Driveへ安全に同期された最新のチェックポイントを自動検出します．

**実行順序**：セクション1〜6で環境復元 → 7でチェックポイント検査 → 8で学習 → 9で保存確認．

**注意**：ランタイムの切断後は環境構築から再実行が必要です．セクション8を同一ランタイムで二重起動しないでください．また，公式設定での学習であり，0.8秒の位相拘束を緩める実験はまだ行いません．

## 1. GPU／システムとGoogle Driveの確認

A100でなくても検証は可能ですが，GPUが割り当てられていることを確認してください．
Google Drive保存先：`MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training`（フォルダー名は完全一致が必要です）．


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, platform, sys, os, json, time, shutil

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training')
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path('/content')
print('作業結果の保存先:', DRIVE)
print('system:', platform.platform())
print('Colab Python:', sys.version.split()[0])
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['bash','-lc','cat /etc/os-release | head -6'], check=False)

## 2. Unitree公式コードと `rsl_rl` を取得

`main` が更新される可能性があるため，毎回コミットハッシュを記録します．既存PCで動いているコードのコミットと同じかどうかは，あとで照合できます．

In [ ]:
from pathlib import Path
import subprocess, json

repo = WORK / 'unitree_rl_gym'
rsl = WORK / 'rsl_rl'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/unitreerobotics/unitree_rl_gym.git', str(repo)], check=True)
if not rsl.exists():
    subprocess.run(['git', 'clone', 'https://github.com/leggedrobotics/rsl_rl.git', str(rsl)], check=True)
# 前回の学習と同じソースコードを再現するためコミットを固定する
subprocess.run(['git', '-C', str(rsl), 'checkout', '2ad79cf0caa85b91721abfe358105f869a784121'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '276801e46c5d433564f24658bac64f254b7d2d4b'], check=True)

rev = {
 'unitree_rl_gym': subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip(),
 'rsl_rl': subprocess.check_output(['git','-C',str(rsl),'rev-parse','HEAD'], text=True).strip()
}
(DRIVE/'source_revisions.json').write_text(json.dumps(rev, indent=2), encoding='utf-8')
print(json.dumps(rev, indent=2))

files = [
 repo/'legged_gym/envs/g1/g1_env.py',
 repo/'legged_gym/envs/g1/g1_config.py',
 repo/'legged_gym/envs/base/legged_robot_config.py',
 repo/'legged_gym/scripts/train.py',
]
for f in files:
    print(('OK  ' if f.exists() else 'MISSING '), f.relative_to(repo))
assert all(f.exists() for f in files), '公式リポジトリの構成が変更されています．'

## 3. Isaac Gym Preview 4の公式配布物を用意（利用許諾）

`IsaacGym_Preview_4_Package.tar` を **NVIDIA公式サイトからご自身でダウンロードし，利用許諾に同意したうえで**，Google Drive の `MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training/` に置いてください．ファイル名が違う場合は下のセルの `ARCHIVE` を変更してください．

公式配布物がないまま次のインストールへは進みません．共有されている非公式ミラーや出所不明のpipパッケージでは代用しません．

In [ ]:
ARCHIVE = DRIVE / 'IsaacGym_Preview_4_Package.tar'
print('期待するファイル:', ARCHIVE)
if not ARCHIVE.is_file():
    print('ファイル未配置です．公式配布サイトから入手したtarファイルをGoogle Driveへアップロードしてください．')
else:
    print('検出:', ARCHIVE.name, '容量 GiB:', round(ARCHIVE.stat().st_size / 1024**3, 2))

## 4. Python 3.8独立環境の作成

Colab本体のPythonを置き換えず，`/content/g1-python38/` に独立環境を作ります．以下は micromamba の公式配布バイナリを利用します．Isaac Gym が要求するOSの違いまでは解決できないため，ここで成功しても学習が動く保証はありません．

In [ ]:
from pathlib import Path
import subprocess, os

MAMBA_BIN = Path('/content/bin/micromamba')
if not MAMBA_BIN.exists():
    subprocess.run(['bash','-lc',
      'mkdir -p /content && curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'], check=True)
PY38 = Path('/content/g1-python38/bin/python')
env = os.environ.copy()
env['MAMBA_ROOT_PREFIX'] = '/content/mamba-root'
if not PY38.exists():
    subprocess.run([str(MAMBA_BIN), 'create','-y','-p','/content/g1-python38',
         '-c','conda-forge','python=3.8','pip'], check=True, env=env)
print('Python 3.8:', subprocess.check_output([str(PY38), '--version'], text=True).strip())

## 5. Isaac Gym／PyTorch／Unitreeライブラリをインストール

公式の `unitree_rl_gym` セットアップに合わせ，Python 3.8，PyTorch 2.3.1 CUDA 12.1，`rsl_rl` v1.0.2 を使用します．NumPyは古いIsaac Gymとの互換性のため 1.23.5 に制限しています．

**重要：** ここでエラーになった場合は次へ進まないでください．無理にバージョン変更を重ねると実験の再現性が失われます．エラー全文を相談してください．

In [ ]:
from pathlib import Path
import subprocess, os

assert ARCHIVE.is_file(), f'Isaac Gym配布物がありません: {ARCHIVE}'
assert PY38.exists(), '先にPython 3.8環境を作成してください．'

EXTRACT = Path('/content/isaacgym_dist')
if not EXTRACT.exists():
    EXTRACT.mkdir(parents=True)
    subprocess.run(['tar','-xf',str(ARCHIVE),'-C',str(EXTRACT)], check=True)

isaac_python_dirs = [p for p in EXTRACT.rglob('python') if (p/'setup.py').is_file() and p.parent.name == 'isaacgym']
assert isaac_python_dirs, '配布物から isaacgym/python/setup.py が見つかりません．構成を確認してください．'
ISAAC_PY_DIR = isaac_python_dirs[0]
print('Isaac Gym Python package:', ISAAC_PY_DIR)

def pip(*args):
    subprocess.run([str(PY38),'-m','pip',*map(str,args)],check=True)

pip('install','--upgrade','pip','setuptools','wheel')
pip('install','numpy==1.23.5','gym==0.23.1','imageio','ninja','tensorboard')
pip('install','torch==2.3.1','torchvision==0.18.1','torchaudio==2.3.1',
    '--index-url','https://download.pytorch.org/whl/cu121')
# 既存GPU版PyTorchを維持し，旧Isaac Gymインストールの依存関係で置き換えない
pip('install','-e',str(ISAAC_PY_DIR),'--no-deps')
pip('install','-e',str(rsl))
pip('install','-e',str(repo))
print('依存ライブラリのインストールが終了しました．まだGPU動作の確認は済んでいません．')

## 6. Isaac Gym の GPU 動作確認

CPU/Pythonのみのimportに成功してもGPU物理演算が動く保証はありません．まずimportとCUDA認識を検査します．次の試験学習で物理エンジンまで確認します．

In [ ]:
import subprocess, os
RUN_ENV = os.environ.copy()
RUN_ENV['LD_LIBRARY_PATH'] = '/content/g1-python38/lib:' + RUN_ENV.get('LD_LIBRARY_PATH','')
RUN_ENV['TORCH_EXTENSIONS_DIR'] = '/content/g1_torch_extensions'
RUN_ENV['PATH'] = str(PY38.parent) + os.pathsep + RUN_ENV.get('PATH', '')
Path(RUN_ENV['TORCH_EXTENSIONS_DIR']).mkdir(exist_ok=True)

probe = r"""
from isaacgym import gymapi
import torch
print('IsaacGym import OK')
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDAを認識していません')
"""
p = subprocess.run([str(PY38),'-c',probe],env=RUN_ENV,capture_output=True,text=True)
print(p.stdout)
print(p.stderr[-6000:])
assert p.returncode == 0, '互換性／依存関係の問題が発生しました．ここで停止してください．'
(DRIVE/'python38_freeze.txt').write_text(subprocess.check_output(
    [str(PY38),'-m','pip','freeze'],text=True,env=RUN_ENV),encoding='utf-8')

In [ ]:
import os
import subprocess
from pathlib import Path

PY38 = Path('/content/g1-python38/bin/python')
BIN38 = PY38.parent

# Ninjaが未インストールの場合も対応
subprocess.run(
    [str(PY38), '-m', 'pip', 'install', 'ninja'],
    check=True
)

# 試験学習の子プロセスに渡す環境変数を修正
RUN_ENV['PATH'] = str(BIN38) + os.pathsep + RUN_ENV.get('PATH', '')

# 子プロセスからNinjaが見えるか確認
check = subprocess.run(
    [
        str(PY38), '-c',
        'import shutil; '
        'from torch.utils.cpp_extension import is_ninja_available; '
        'print("Ninja path:", shutil.which("ninja")); '
        'print("Ninja available:", is_ninja_available())'
    ],
    env=RUN_ENV,
    capture_output=True,
    text=True
)

print(check.stdout)
print(check.stderr[-2000:])
assert check.returncode == 0
assert 'Ninja available: True' in check.stdout

## 7．継続学習チェックポイントの自動検出

Google Driveの`full_logs/g1`以下から，**前回の1,000反復チェックポイント**と，このノートブックで作成された`colab_continue_to_10000`実行フォルダーのチェックポイントのみを候補として探索します．

利用可能な反復数のうち最大のモデルを選び，モデル重み・オプティマイザー状態・反復数を検査します．このセルに`再開準備OK`と表示されてから次へ進んでください．

In [ ]:
import re, shutil, subprocess, os
from pathlib import Path

TARGET_ITER = 10000
BASE_RUN = 'Oct23_16-15-05_colab_resume_500_to_1000'
CONTINUE_SUFFIX = '_colab_continue_to_10000'
DRIVE_G1_LOGS = DRIVE / 'full_logs' / 'g1'
LOCAL_G1_LOGS = repo / 'logs' / 'g1'
assert DRIVE_G1_LOGS.is_dir(), f'ログ保存先がありません: {DRIVE_G1_LOGS}'

candidate_runs = [DRIVE_G1_LOGS / BASE_RUN]
candidate_runs += sorted(p for p in DRIVE_G1_LOGS.iterdir()
                         if p.is_dir() and p.name.endswith(CONTINUE_SUFFIX))

candidates = []
for run in candidate_runs:
    if not run.is_dir():
        continue
    for model in run.glob('model_*.pt'):
        match = re.fullmatch(r'model_(\d+)\.pt', model.name)
        if match:
            iteration = int(match.group(1))
            if iteration >= 1000 and iteration <= TARGET_ITER and model.stat().st_size > 100_000:
                candidates.append((iteration, model))

assert candidates, '再開元のmodel_1000.ptが見つかりません．Google Driveの保存先を確認してください．'
SOURCE_ITER, SOURCE_MODEL = max(candidates, key=lambda z: (z[0], z[1].stat().st_size))
SOURCE_RUN = SOURCE_MODEL.parent.name
print('再開元:', SOURCE_MODEL)
print('チェックポイント反復数:', SOURCE_ITER)
print('残り:', TARGET_ITER-SOURCE_ITER, '反復')

assert PY38.is_file() and repo.is_dir() and rsl.is_dir(), '先に環境構築セルを実行してください．'
assert 'RUN_ENV' in globals(), 'セクション6まで実行してください．'
expected = {
 'unitree_rl_gym': '276801e46c5d433564f24658bac64f254b7d2d4b',
 'rsl_rl': '2ad79cf0caa85b91721abfe358105f869a784121',
}
for name, path in [('unitree_rl_gym',repo),('rsl_rl',rsl)]:
    actual = subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'], text=True).strip()
    assert actual == expected[name], f'{name}のコミットが前回の学習と異なります: {actual}'

LOCAL_RUN = LOCAL_G1_LOGS / SOURCE_RUN
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL = LOCAL_RUN / SOURCE_MODEL.name
shutil.copy2(SOURCE_MODEL, LOCAL_MODEL)

# 自分のGoogle Driveで作成した信頼できるモデルに限って読み込む
check_code = '\n'.join([
 'import torch, sys',
 'obj = torch.load(sys.argv[1], map_location="cpu")',
 'print("checkpoint iter:", obj.get("iter"))',
 'print("optimizer present:", "optimizer_state_dict" in obj)',
 'print("model weights present:", "model_state_dict" in obj)',
 f'assert obj.get("iter") == {SOURCE_ITER}',
 'assert "optimizer_state_dict" in obj and "model_state_dict" in obj',
])
check = subprocess.run([str(PY38),'-c',check_code,str(LOCAL_MODEL)],
                       env=RUN_ENV, capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    print(check.stderr[-4000:])
assert check.returncode == 0, 'チェックポイント内容の検証に失敗しました．学習を開始しないでください．'
if SOURCE_ITER >= TARGET_ITER:
    print('すでに10,000反復に到達しています．セクション8は実行不要です．')
else:
    print('再開準備OK: 累積', SOURCE_ITER, '→', TARGET_ITER)


## 8．連続学習（10,000反復まで）

上のセルに「再開準備OK」と表示された場合のみ実行してください．

学習反復数は**累積目標 − チェックポイント反復数**として自動計算します．例：1,000からなら`--max_iterations=9000`です．今回の固定したRSL-RLでは，この引数は再開後に追加する反復数です（過去の継続学習で確認済み）．

チェックポイント自体は学習コードの保存間隔（前回は50反復ごと）に応じて生成されます．以下のセルでは**約2分ごとに完成済みチェックポイントをGoogle Driveに安全に同期**します．転送中の`.pt`を完成済みモデルとして扱わないよう，一時ファイルから置き換えます．

Colab Proでもランタイムの連続稼働時間は保証されません．切断すると最後に同期された反復から再開します．途中の学習ログは`DRIVE/continue_to_10000_output.txt`にあります．

In [ ]:
import time, subprocess, shutil, os, re
from pathlib import Path

assert SOURCE_ITER < TARGET_ITER, '学習目標にすでに到達しています．'
assert LOCAL_MODEL.is_file(), 'まずセクション7を実行してください．'
ADDITIONAL_ITERATIONS = TARGET_ITER - SOURCE_ITER
RUN_SUFFIX = 'colab_continue_to_10000'
LOGFILE = DRIVE / 'continue_to_10000_output.txt'
LOCAL_G1_LOGS.mkdir(parents=True, exist_ok=True)
DRIVE_G1_LOGS.mkdir(parents=True, exist_ok=True)

# このランタイムで同じ継続実行を開始していれば二重起動を防ぐ
assert not list(LOCAL_G1_LOGS.glob('*_'+RUN_SUFFIX)), (
    'このランタイムには継続学習の実行フォルダーがすでにあります．二重起動しないでください．')

cmd = [
    str(PY38), 'legged_gym/scripts/train.py', '--task=g1', '--headless',
    '--num_envs=4096', f'--max_iterations={ADDITIONAL_ITERATIONS}',
    '--seed=42', '--resume', f'--load_run={SOURCE_RUN}',
    f'--checkpoint={SOURCE_ITER}', f'--run_name={RUN_SUFFIX}'
]
env = RUN_ENV.copy()
env['PATH'] = str(PY38.parent) + os.pathsep + env.get('PATH','')

# 前回の実行ログを上書きしないよう，時刻入りの新ログを作成
LOGFILE = DRIVE / f'continue_to_10000_from_{SOURCE_ITER}_{time.strftime("%Y%m%d_%H%M%S")}.txt'

# 50反復おきのチェックポイントをDriveへコピーする（コピー先では一時名で作成してから確定）
def sync_completed_checkpoints(final=False):
    runs = sorted(LOCAL_G1_LOGS.glob('*_'+RUN_SUFFIX), key=lambda p: p.stat().st_mtime)
    if not runs:
        return (None, [])
    run = runs[-1]
    dest = DRIVE_G1_LOGS / run.name
    dest.mkdir(parents=True, exist_ok=True)
    copied = []
    for src in sorted(run.glob('model_*.pt'), key=lambda p: p.stat().st_mtime):
        stat1 = src.stat()
        # 書き込み途中である可能性のあるファイルはコピーしない
        if time.time() - stat1.st_mtime < 30:
            continue
        dst = dest / src.name
        if dst.exists() and dst.stat().st_size == stat1.st_size:
            continue
        temp = dest / (src.name + '.copying')
        try:
            shutil.copy2(src, temp)
            stat2 = src.stat()
            if stat1.st_size != stat2.st_size or stat1.st_mtime_ns != stat2.st_mtime_ns:
                temp.unlink(missing_ok=True)
                print('書き込み変化を検知，次回同期:', src.name)
                continue
            os.replace(temp, dst)
            copied.append(src.name)
        except Exception:
            temp.unlink(missing_ok=True)
            raise
    # 学習ログやハイパーパラメータ等は最終時点で退避
    if final:
        for file in run.iterdir():
            if file.is_file() and not file.name.startswith('model_') and not file.name.endswith('.copying'):
                try:
                    shutil.copy2(file, dest/file.name)
                except OSError as e:
                    print('補助ファイルを保存できませんでした:', file.name, e)
    return run.name, copied

print('再開元:', SOURCE_MODEL)
print('累積目標:', TARGET_ITER, '追加反復:', ADDITIONAL_ITERATIONS)
print('実行コマンド:', ' '.join(cmd))
print('実行ログ:', LOGFILE)
started = time.time()
with LOGFILE.open('w',encoding='utf-8') as f:
    proc = subprocess.Popen(cmd, cwd=str(repo), env=env,
                            stdout=f, stderr=subprocess.STDOUT)
    last_synced = 0
    try:
        while proc.poll() is None:
            time.sleep(15)
            if time.time()-last_synced >= 120:
                name, copied = sync_completed_checkpoints()
                print('Drive同期:', name or '実行フォルダー生成待ち', '新規:', copied[-4:])
                last_synced = time.time()
    except KeyboardInterrupt:
        print('ユーザーによる中断: 学習プロセスを停止します．')
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise
    finally:
        # 直近の.ptのmtimeが30秒以内なら，書き込み完了を待って最終同期
        if proc.poll() is not None:
            time.sleep(31)
        name, copied = sync_completed_checkpoints(final=True)
        print('最終Drive同期:', name, '新規:', copied)

print('終了コード:',proc.returncode)
print('経過分:',round((time.time()-started)/60,1))
print('ログ末尾:\n',LOGFILE.read_text(encoding='utf-8',errors='replace')[-6000:])
assert proc.returncode == 0, '学習が終了コード0以外で停止しました．Drive側の最終チェックポイントを確認してください．'


## 9．保存済みチェックポイントと学習進捗の確認

学習中の別のセルから確認することもできます．`model_10000.pt`が見つかるまでは，最新モデルの反復数を表示します．

In [ ]:
import re
from pathlib import Path
root = DRIVE / 'full_logs' / 'g1'
base_run = 'Oct23_16-15-05_colab_resume_500_to_1000'
runs = [root / base_run] + sorted(p for p in root.glob('*_colab_continue_to_10000') if p.is_dir())
seen = []
for run in runs:
    if not run.is_dir():
        continue
    models = []
    for p in run.glob('model_*.pt'):
        match = re.fullmatch(r'model_(\d+)\.pt',p.name)
        if match and p.stat().st_size > 100_000:
            models.append((int(match.group(1)),p))
    if models:
        best = max(models, key=lambda x:x[0])
        seen.append(best)
        print(run.name, ':', len(models), '個／最新', best[1].name)
if seen:
    final_iter, final_path = max(seen, key=lambda x:x[0])
    print('\n最新チェックポイント:', final_path)
    print('累積進捗:', final_iter, '/ 10000 (',round(100*final_iter/10000,1),'%)')
    print('model_10000.pt:',final_iter == 10000)
else:
    print('チェックポイントがまだありません．保存先を確認してください．')
